In [1]:
%run /Users/manojrammopati/Downloads/bupa_insurance_project/01_Bronze_Layer/Notebooks/_01_bronze_container_connect.ipynb

25/11/28 17:42:43 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/11/28 17:42:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f7718ed8-cdf8-48bc-a420-7183e518f2d4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in c

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:
import sys
import importlib
import utils_silver
from utils_silver import *

# Add the Silver layer directory to sys.path and import the utils module

silver_dir = "/Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
importlib.reload(utils_silver)


paths_bronze, paths_silver = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
# Optionally bring helper names into the notebook namespace (avoid if you prefer module namespace)

print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# Cell 1 — Imports + Path Setup

In [14]:
# Cell 1 — Imports + Paths

from pyspark.sql import functions as F
from pyspark.sql.types import *
from utils_silver import (
    build_paths, enforce_schema, quarantine, drop_dupes_keep_latest,
    dq_expect, dq_left_anti_ref, write_metric
)

storage_account = "clientdatastorage"
paths_bronze, paths_silver = build_paths(storage_account)

BRONZE_PROVIDERS_PATH = paths_bronze["providers"]
SILVER_PROVIDERS_PATH = paths_silver["providers"]

print("Bronze Providers:", BRONZE_PROVIDERS_PATH)
print("Silver Providers:", SILVER_PROVIDERS_PATH)

# make sure the logical DB exists
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_silver")


Bronze Providers: abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers
Silver Providers: abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers


DataFrame[]

# Cell 2 — Read Bronze + Schema Enforcement

In [15]:
# Cell 2 — Read Bronze Providers + Enforce Schema

providers_bz = spark.read.format("delta").load(BRONZE_PROVIDERS_PATH)

print("Bronze schema:")
providers_bz.printSchema()

provider_schema = StructType([
    StructField("Provider_ID", StringType()),
    StructField("Fraud_Flag", IntegerType())
])

providers = enforce_schema(providers_bz, provider_schema)

providers.show(5)
print("Bronze count:", providers.count())


Bronze schema:
root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)

+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|   PRV51262|         0|
|   PRV51606|         0|
|   PRV51657|         0|
|   PRV51665|         0|
|   PRV51749|         0|
+-----------+----------+
only showing top 5 rows

Bronze count: 5410


# Cell 3 — Primary Key Validation

In [16]:
# Cell 3 — PK Validation

dq_expect(
    providers,
    "pk_not_null",
    "Provider_ID IS NOT NULL",
    "error",
    "providers",
    paths_silver
)

print("After PK validation:", providers.count())


✅ DQ PASS [providers] pk_not_null
After PK validation: 5410


# Cell 4 — Fraud Flag Normalisation + DQ

In [17]:
# Cell 4 — Fraud Flag normalization

# Normalize Fraud_Flag → int 0/1
providers_clean = (
    providers
    .withColumn(
        "Fraud_Flag",
        F.when(F.col("Fraud_Flag").isin(1, "1", "Y", "Yes", "YES", "True", "true"), 1)
         .otherwise(0)
         .cast("int")
    )
)

# Validate flag integrity
dq_expect(
    providers_clean,
    "fraud_flag_valid",
    "Fraud_Flag IN (0,1)",
    "error",
    "providers",
    paths_silver
)

print("After Fraud_Flag validation:", providers_clean.count())
providers_clean.show(5)


✅ DQ PASS [providers] fraud_flag_valid
After Fraud_Flag validation: 5410
+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|   PRV51262|         0|
|   PRV51606|         0|
|   PRV51657|         0|
|   PRV51665|         0|
|   PRV51749|         0|
+-----------+----------+
only showing top 5 rows



# Cell 5 — Deduplication

In [18]:
# Cell 5 — Deduplication (latest record wins)

providers_dedup = drop_dupes_keep_latest(
    providers_clean,
    key_cols=["Provider_ID"],
    order_desc_cols=["Fraud_Flag"]    # if duplicates exist, fraud=1 takes precedence
)

print("After dedupe:", providers_dedup.count())
providers_dedup.show(5)


After dedupe: 5410
+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|   PRV51001|         0|
|   PRV51003|         1|
|   PRV51004|         0|
|   PRV51005|         1|
|   PRV51007|         0|
+-----------+----------+
only showing top 5 rows



# Cell 6 — Optional Reference Dimension (Provider Fraud)

In [19]:
# Cell 6 — Optional reference dimension

ref_path = paths_silver["_reference"] + "/dim_provider_fraud"

dim_provider_fraud = spark.createDataFrame(
    [(0,), (1,)],
    "Fraud_Flag INT"
)

(dim_provider_fraud.write
     .format("delta")
     .mode("overwrite")
     .save(ref_path))

print("Created reference table at:", ref_path)


Created reference table at: abfss://silverdata@clientdatastorage.dfs.core.windows.net/reference/dim_provider_fraud


# Cell 7 — Reference Validation

In [20]:
# Cell 7 — Validate Fraud_Flag matches reference table

dim_provider_fraud = spark.read.format("delta").load(ref_path)

dq_left_anti_ref(
    providers_dedup,
    dim_provider_fraud,
    key_col="Fraud_Flag",
    ref_col="Fraud_Flag",
    name="fraud_flag_in_reference",
    severity="error",
    table_label="providers",
    paths_silver=paths_silver
)

providers_valid = providers_dedup


✅ DQ PASS [providers] fraud_flag_in_reference


# Cell 8 — Write Silver Providers

In [21]:
# Cell 8 — Write Silver Providers

(providers_valid.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PROVIDERS_PATH))

print("Silver providers written to:", SILVER_PROVIDERS_PATH)

# Register table for SQL/BI
spark.sql("""
CREATE TABLE IF NOT EXISTS bupa_silver.providers
USING DELTA
LOCATION '{}'
""".format(SILVER_PROVIDERS_PATH))

print("Registered table: bupa_silver.providers")


Silver providers written to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers
Registered table: bupa_silver.providers


# Cell 9 — Metrics Logging

In [22]:
# Cell 9 — Metrics Logging

write_metric(
    spark,
    "rowcount_providers_silver",
    providers_valid.count(),
    "providers_silver",
    paths_silver
)

write_metric(
    spark,
    "distinct_provider_ids",
    providers_valid.select("Provider_ID").distinct().count(),
    "providers_silver",
    paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)


[METRIC] rowcount_providers_silver=5410 ctx=providers_silver
[METRIC] distinct_provider_ids=5410 ctx=providers_silver


+--------------------------+------+----------------+--------------------------+
|metric                    |value |context         |ts                        |
+--------------------------+------+----------------+--------------------------+
|distinct_provider_ids     |5410  |providers_silver|2025-11-29 00:42:09.375763|
|rowcount_providers_silver |5410  |providers_silver|2025-11-29 00:42:04.874106|
|distinct_claim_ids        |558211|claims_silver   |2025-11-28 15:31:16.213598|
|rowcount_claims_silver    |558211|claims_silver   |2025-11-28 15:30:54.237362|
|distinct_member_ids       |381109|members_silver  |2025-11-28 14:04:23.229343|
|rowcount_members_silver   |381109|members_silver  |2025-11-28 14:04:14.937709|
|distinct_policy_ids_silver|381109|policies_silver |2025-11-28 10:36:21.144919|
|rowcount_policies_silver  |381109|policies_silver |2025-11-28 10:36:03.700764|
|distinct_policy_ids       |381109|policies_silver |2025-11-25 01:31:29.412102|
|rowcount_policies_silver  |381109|polic

# Cell 10 — Final Output Check

In [23]:
# Cell 10 — Final Silver Check

providers_silver = spark.read.format("delta").load(SILVER_PROVIDERS_PATH)

print("Silver Providers Count:", providers_silver.count())
providers_silver.show(5)
providers_silver.printSchema()


Silver Providers Count: 5410
+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|   PRV51001|         0|
|   PRV51003|         1|
|   PRV51004|         0|
|   PRV51005|         1|
|   PRV51007|         0|
+-----------+----------+
only showing top 5 rows

root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)



# ⭐ A–D: SILVER PROVIDERS ANALYSIS CELLS (Same Style as Policies/Members/Claims)

# A) Bronze vs Silver Comparison

In [24]:
# A) Bronze vs Silver comparison

br = providers
sv = providers_silver

bronze_cols = set(br.columns)
silver_cols = set(sv.columns)

print("New columns in Silver:", silver_cols - bronze_cols)
print("Dropped columns:", bronze_cols - silver_cols)
print("Common columns:", bronze_cols & silver_cols)


New columns in Silver: set()
Dropped columns: set()
Common columns: {'Fraud_Flag', 'Provider_ID'}


# B) Per-Column Change Report

In [25]:
# B) Per-column difference report

common_cols = sorted(bronze_cols & silver_cols)

for col in common_cols:
    n_diff = (
        br.alias("b").join(sv.alias("s"), "Provider_ID", "outer")
          .filter((F.col(f"b.{col}") != F.col(f"s.{col}")) |
                  (F.col(f"b.{col}").isNull() & F.col(f"s.{col}").isNotNull()) |
                  (F.col(f"b.{col}").isNotNull() & F.col(f"s.{col}").isNull()))
    ).count()
    
    print(f"Column '{col}' changed in {n_diff} rows")


Column 'Fraud_Flag' changed in 0 rows
Column 'Provider_ID' changed in 0 rows


# C) Summary DQ Improvements

In [26]:
# C) DQ improvements overview

print("Bronze Null Provider IDs:", br.filter("Provider_ID IS NULL").count())
print("Silver Null Provider IDs:", sv.filter("Provider_ID IS NULL").count())

print("Bronze Non-Binary Fraud_Flag:", br.filter("Fraud_Flag NOT IN (0,1)").count())
print("Silver Non-Binary Fraud_Flag:", sv.filter("Fraud_Flag NOT IN (0,1)").count())


Bronze Null Provider IDs: 0
Silver Null Provider IDs: 0
Bronze Non-Binary Fraud_Flag: 0
Silver Non-Binary Fraud_Flag: 0


# D) Provider Dimension Coverage

In [ ]:
# D) Check Fraud_Flag coverage vs dimension

dim = dim_provider_fraud

dq_left_anti_ref(
    sv, dim, "Fraud_Flag", "Fraud_Flag",
    name="fraud_flag_dimension_check",
    severity="warn",
    table_label="providers",
    paths_silver=paths_silver
)


✅ DQ PASS [providers] fraud_flag_dimension_check


25/11/29 03:47:07 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 929540 ms exceeds timeout 120000 ms
25/11/29 03:47:08 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/29 03:47:11 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$